# Train WallFloorNet on Colab (reproduces `weights/` and `results/`)
**Runtime → Change runtime type → T4 GPU**, then **Run all**. Takes roughly 1.5 h end to end:
dataset download and prep ~10 min, training ~60–75 min, evaluation ~10 min.
At the end, `trained_artifacts.zip` (weights, ONNX/INT8 models, metrics, gallery) downloads automatically.
Unzip it into the repository root and commit.

In [ ]:
import os, sys, subprocess, torch
assert torch.cuda.is_available(), "Switch the runtime to a GPU (Runtime > Change runtime type > T4 GPU)"
print(torch.cuda.get_device_name(0), "| CPUs:", os.cpu_count())
REPO_URL = "https://github.com/shlokdhanokar/nirwana-wall-floor-segmentation"
if not os.path.isdir("repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
os.chdir("repo")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime", "onnx"], check=True)

In [ ]:
%%time
# ADE20K SceneParsing (MIT CSAIL): ~1 GB
if not os.path.isdir("data/ADEChallengeData2016"):
    os.makedirs("data", exist_ok=True)
    !wget -q -O data/ade.zip http://data.csail.mit.edu/places/ADEchallenge/ADEChallengeData2016.zip
    !unzip -q data/ade.zip -d data && rm data/ade.zip
!python -m wallfloor.prepare --ade-root data/ADEChallengeData2016 --out data/cache --workers {os.cpu_count()}

In [ ]:
EPOCHS = 30
# encoder from ImageNet classification weights (disclosed); add --no-pretrained-encoder for the from-scratch ablation
!python -m wallfloor.train --cache data/cache --out weights/wallfloor_mbv3.pt --epochs {EPOCHS} --batch 16 --workers {os.cpu_count()} --loader process

In [ ]:
!python -m wallfloor.export --weights weights/wallfloor_mbv3.pt --calib data/cache   # FP32 ONNX + INT8 (percentile, 16 calib images)
!ls -la weights

In [ ]:
%%time
# quality on the GPU; latency on this runtime's CPU (threads = all vCPUs)
!python scripts/evaluate.py --weights weights/wallfloor_mbv3.pt --threads {os.cpu_count()}
!python scripts/make_gallery.py --weights weights/wallfloor_mbv3.pt --n 300

In [ ]:
!zip -q -r /content/trained_artifacts.zip weights results
try:
    from google.colab import files
    files.download("/content/trained_artifacts.zip")
except Exception as e:
    print("Download /content/trained_artifacts.zip from the file browser:", e)